# Association between diagnoses recorded at baseline and AgeAccel (odds ratios)

This notebook estimates the odds ratio (OR) of a diagnosis recorded at baseline per 1 SD increase in AgeAccel.

**Model**: for each of 8 pre-specified diagnoses, an unpenalized logistic regression `diagnosis ~ AgeAccel (per 1 SD) + age + sex` is fitted and compared with the nested model `diagnosis ~ age + sex` by a likelihood-ratio test (LRT, 1 d.f.); the OR and Wald 95% CI are reported. This primary analysis is **not adjusted for BMI**.

**Cohort and scores**: the primary analysis uses the shared four-omics cohort of Result-04, with Transcriptomics, Lipidomics, Gut Microbiome, Oral Microbiome and the stacked four-omics AgeAccel (age-residualized predicted age difference, standardized to mean 0 and SD 1 within the analysis cohort). Metabolomics uses its largest available cohort (not the same participants as the primary cohort), is marked with † and serves as a descriptive reference only.

**Multiple comparisons**: the 40 tests of the primary analysis (8 diagnoses × 5 scores) form one BH-FDR family; the 8 Metabolomics tests are adjusted separately.

**Scope**: cross-sectional associations. The outcomes are ICD-11 diagnoses recorded in the baseline questionnaire and are not presented as prospective prediction or causal effects.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'disease_odds_ratio.py').exists():
    NOTEBOOK_DIR = Path('02-papers/Result-05').resolve()
sys.path.insert(0, str(NOTEBOOK_DIR))

MEDICAL_CONDITIONS_PATH = Path(r'/home/ec2-user/studies/hpp_datasets/medical_conditions/medical_conditions.parquet')
POPULATION_PATH = Path(r'/home/ec2-user/studies/hpp_datasets/population/population.parquet')
OUTPUT_DIR = NOTEBOOK_DIR / 'outputs'
FIGURE_DIR = NOTEBOOK_DIR / 'figures'

## Model fitting

Diagnoses, clock scores and sex are read and 48 models are fitted (40 in the primary cohort + 8 for Metabolomics); results are saved to `outputs/disease_or_results.csv`. The cell prints a summary table of ORs; `*` marks BH-FDR q < 0.05.

In [ ]:
from disease_odds_ratio import run_analysis

df_or = run_analysis(
    medical_path=MEDICAL_CONDITIONS_PATH,
    population_path=POPULATION_PATH,
    output_dir=OUTPUT_DIR,
)

## Plotting

Plotting reads `outputs/disease_or_results.csv` only and does not refit the models. With existing results, run this cell in a fresh kernel or run `python disease_odds_ratio.py --plot-only` in a terminal.

- `figures/disease_or_main_stack.*`: main-figure panel, stacked four-omics AgeAccel, one row per diagnosis (89 mm single-column width). Filled points have q < 0.05; case numbers are in parentheses.
- `figures/disease_or_extended_all_scores.*`: Extended Data, 2 × 4 grid, all 5 primary scores and Metabolomics† for each diagnosis (183 mm double-column width).

Diagnoses follow the pre-specified order by default; with `sort_by_or=True` they are ordered by decreasing stacked four-omics OR.

In [ ]:
from disease_odds_ratio import run_plots

_ = run_plots(
    results_path=OUTPUT_DIR / 'disease_or_results.csv',
    figure_dir=FIGURE_DIR,
    sort_by_or=False,
)

## Results table

Main columns: `odds_ratio`, `or_ci_low`, `or_ci_high` (OR per 1 SD AgeAccel and 95% CI); `lr_p` (likelihood-ratio test P value, the primary test); `wald_p` (Wald test P value, for reference); `q_value` (BH-FDR); `score_sd_years` (years corresponding to 1 SD of AgeAccel, for converting the OR to a per-year effect); `fit_status` (whether the model converged).

In [ ]:
import pandas as pd

df_or = pd.read_csv(OUTPUT_DIR / 'disease_or_results.csv')
cols = ['outcome', 'score', 'cohort', 'n', 'n_cases', 'score_sd_years',
        'odds_ratio', 'or_ci_low', 'or_ci_high', 'lr_p', 'q_value', 'fit_status']
display(df_or[cols].round(4))
display(df_or[df_or['fdr_significant']][cols].round(4))